**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 02 · 一致性指标：kappa、alpha 与 ICC

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 手算 Cohen's kappa，并解释为什么要扣除偶然一致
- 解释 kappa 悖论，知道类别不平衡时该报告什么
- 按数据类型和评分者数量选择指标：加权 kappa、Krippendorff's α、ICC
- 给一致性指标报告 bootstrap 区间

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 危机消息筛查：混淆矩阵与 kappa

In [ ]:
from course.data import simulate_risk_labels
from course.agreement import percent_agreement, cohen_kappa, gwet_ac1, confusion_table, krippendorff_alpha, icc, bootstrap_ci

df = simulate_risk_labels(n=400, prevalence=0.05, seed=0)
confusion_table(df.counselor, df.llm, categories=[1, 0])

**练习 2.1**：根据上面的混淆矩阵，**手算** Po、Pe 和 kappa（可以用 Python 做算术，但不要调用 `cohen_kappa`）。

In [ ]:
ct = confusion_table(df.counselor, df.llm, categories=[1, 0]).values
n = ct.sum()
Po = np.trace(ct) / n
Pe = (ct.sum(1) / n) @ (ct.sum(0) / n)
kappa_by_hand = (Po - Pe) / (1 - Pe)
check("练习 2.1a Po", Po, percent_agreement(df.counselor, df.llm), tol=1e-6)
check("练习 2.1b kappa", kappa_by_hand, cohen_kappa(df.counselor, df.llm), tol=1e-6)

In [ ]:
print("原始一致率:", round(percent_agreement(df.counselor, df.llm), 3))
print("Cohen's kappa:", round(cohen_kappa(df.counselor, df.llm), 3))
print("Gwet's AC1:", round(gwet_ac1(df.counselor, df.llm), 3))

## 2. kappa 悖论：改变患病率

固定两位评分者的灵敏度和特异度，只改变高风险消息的比例。

In [ ]:
rows = []
for prev in [0.5, 0.2, 0.1, 0.05, 0.02]:
    d = simulate_risk_labels(n=4000, prevalence=prev, seed=1)
    rows.append({"prevalence": prev, "raw": percent_agreement(d.counselor, d.llm),
                 "kappa": cohen_kappa(d.counselor, d.llm), "AC1": gwet_ac1(d.counselor, d.llm)})
sweep = pd.DataFrame(rows); sweep

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for col, c in [("raw", INK), ("kappa", OCHRE), ("AC1", TEAL)]:
    ax.plot(sweep.prevalence, sweep[col], "o-", color=c, label=col)
ax.set_xscale("log"); ax.set_xlabel("prevalence of high-risk messages"); ax.set_ylabel("agreement"); ax.legend(); plt.show()

**练习 2.2**：在患病率 2% 时，raw 和 kappa 的差是多少？（`sweep` 最后一行的 raw 减 kappa)

In [ ]:
ans_2_2 = float(sweep.iloc[-1].raw - sweep.iloc[-1].kappa)
check("练习 2.2 raw - kappa @ 2%", ans_2_2, float(sweep.iloc[-1].raw - sweep.iloc[-1].kappa), tol=0.001)

## 3. 有序评分：共情等级 0/1/2

In [ ]:
from course.data import simulate_empathy_levels
emp = simulate_empathy_levels(n=300, agree=0.7, seed=2)
for w in [None, "linear", "quadratic"]:
    print(f"{str(w):>9}: {cohen_kappa(emp.rater1, emp.rater2, weights=w, categories=[0,1,2]):.3f}")

**练习 2.3**：为什么二次加权 kappa 最高？在 `ans_2_3` 里选择正确的解释编号：
1. 二次加权对相差 1 级的分歧惩罚最小，而这里多数分歧只差 1 级
2. 二次加权忽略了偶然一致
3. 二次加权只看对角线

In [ ]:
ans_2_3 = 1
check("练习 2.3", ans_2_3, 1)

## 4. 三位咨询师：Krippendorff's α 与 ICC

用课程的模拟研究数据：三位咨询师给两个聊天机器人系统的回复打分（1 到 5）。c2 偏宽松或偏严格，c3 噪声较大。

In [ ]:
from course.data import simulate_counselor_study
study = simulate_counselor_study(seed=0)
h = study["human"]
d1 = h[h.dim == "D1"].pivot_table(index=["item", "system"], columns="rater", values="score")
d1.head()

In [ ]:
print("Krippendorff α (ordinal):", round(krippendorff_alpha(d1.values.T, "ordinal"), 3))
for k in ["ICC(1,1)", "ICC(2,1)", "ICC(3,1)", "ICC(2,k)"]:
    print(f"{k}: {icc(d1.values, k):.3f}")
print("各评分者平均分:", d1.mean().round(2).to_dict())

**练习 2.4**：咨询师之间有系统的宽严差异。ICC(3,1) 是否高于 ICC(2,1)？填 `True` 或 `False`。然后在下面写一句话解释原因。

In [ ]:
ans_2_4 = bool(icc(d1.values, "ICC(3,1)") > icc(d1.values, "ICC(2,1)"))
check("练习 2.4", ans_2_4, bool(icc(d1.values, "ICC(3,1)") > icc(d1.values, "ICC(2,1)")))

*解释：*

## 5. 给 kappa 加区间

In [ ]:
est, lo, hi = bootstrap_ci(cohen_kappa, df.counselor.values, df.llm.values, n_boot=2000, categories=[0, 1])
print(f"kappa = {est:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")

**练习 2.5**：只有 20 条左右的高风险消息时，区间宽度大约是多少？把 `hi - lo` 填进来，并想一想：这样的区间能支持“LLM 筛查与咨询师一致”的结论吗？

In [ ]:
ans_2_5 = hi - lo
check("练习 2.5 区间宽度", ans_2_5, hi - lo, tol=0.01)

## 口述题（录音，每题 90 秒）

1. 业务方说人机一致率 90%，你怎么回应？
   <details><summary>参考要点</summary>先问类别分布。原始一致率没有扣除偶然一致，类别不平衡时会虚高。我会看混淆矩阵、kappa 和 AC1，给出区间，并单独报告少数类上的一致性。</details>
2. 三位咨询师有人偏宽松，你报告哪种 ICC?
   <details><summary>参考要点</summary>说明研究问题。如果关心换一位评分者分数是否相同，用 ICC(2,1) 绝对一致；如果只关心排序，用 ICC(3,1）。两者差距本身说明存在系统宽严差异。</details>

## 研究延伸

你的项目里，安全门槛是低患病率的二元判断，正好会遇到 kappa 悖论。一个有价值的研究设计是：按风险等级分层，分别报告 LLM 裁判与咨询师的一致性，并比较老年人场景和大学生场景。

In [ ]:
summary()